# 03 — Multiclass Classification: Readmission Timing

**Task:** Predict the *timing* of readmission: **No** readmission / **<30 days** / **>30 days**

**Target:** `readmitted` — categorical: `NO`, `<30`, `>30`

This notebook covers:
1. Feature selection for multiclass (k=19)
2. Class imbalance handling with SMOTE
3. Model comparison (Decision Tree, Random Forest, AdaBoost, MLP, Naive Bayes)
4. Final evaluation with macro F1 and per-class breakdown
5. Feature importance analysis

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sys
sys.path.insert(0, '..')

from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_classif
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import f1_score, classification_report
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

from src.preprocessing import DataPreprocessor
from src.feature_engineering import FeatureEngineer
from src.evaluate import ModelEvaluator

sns.set_style('whitegrid')
SEED = 42
print('Ready.')

## 1. Load and Prepare Data

In [ ]:
train = pd.read_csv('../data/train.csv')
test  = pd.read_csv('../data/test.csv')

TARGET = 'readmitted'
DROP_TARGETS = [c for c in ('readmitted', 'readmitted_binary') if c in train.columns]

y_train = train[TARGET]
y_test  = test[TARGET]

X_train_raw = train.drop(columns=DROP_TARGETS)
X_test_raw  = test.drop(columns=DROP_TARGETS)

evaluator = ModelEvaluator(task='multiclass')
fig = evaluator.plot_class_distribution(y_train, title='Multiclass Target Distribution (Train)')
plt.show()

print(y_train.value_counts(normalize=True).round(3))

In [ ]:
preprocessor = DataPreprocessor()
preprocessor.fit(X_train_raw)
X_train_proc = preprocessor.transform(X_train_raw)
X_test_proc  = preprocessor.transform(X_test_raw)

engineer = FeatureEngineer()
engineer.fit(X_train_proc)
X_train = engineer.transform(X_train_proc)
X_test  = engineer.transform(X_test_proc)

print(f'Feature matrix: {X_train.shape}')

## 2. Feature Selection (k=19)

In [ ]:
K_BEST = 19
mi_scores = mutual_info_classif(X_train, y_train, random_state=SEED)
mi_df = pd.DataFrame({'feature': X_train.columns, 'MI': mi_scores}).sort_values('MI', ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
top20 = mi_df.head(20)
ax.barh(top20['feature'], top20['MI'], color='#DD8452')
ax.set_xlabel('Mutual Information Score')
ax.set_title('Top 20 Features — Mutual Information (Multiclass Task)')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

selector = SelectKBest(score_func=f_classif, k=K_BEST)
selector.fit(X_train, y_train)
selected_features = X_train.columns[selector.get_support()].tolist()
print(f'Selected {K_BEST} features: {selected_features}')

## 3. Model Comparison

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
X_sel = X_train[selected_features]

models = {
    'Decision Tree': ImbPipeline([
        ('smote', SMOTE(random_state=SEED)),
        ('clf', DecisionTreeClassifier(max_depth=10, random_state=SEED))
    ]),
    'Random Forest': ImbPipeline([
        ('smote', SMOTE(random_state=SEED)),
        ('clf', RandomForestClassifier(n_estimators=200, random_state=SEED, n_jobs=-1))
    ]),
    'AdaBoost': ImbPipeline([
        ('smote', SMOTE(random_state=SEED)),
        ('clf', CalibratedClassifierCV(
            AdaBoostClassifier(n_estimators=100, learning_rate=0.5, random_state=SEED), cv=3
        ))
    ]),
    'MLP': ImbPipeline([
        ('smote', SMOTE(random_state=SEED)),
        ('clf', MLPClassifier(hidden_layer_sizes=(100,50), max_iter=300, random_state=SEED))
    ]),
    'Naive Bayes': ImbPipeline([
        ('smote', SMOTE(random_state=SEED)),
        ('clf', GaussianNB())
    ]),
}

results = {}
for name, model in models.items():
    scores = cross_val_score(model, X_sel.values, y_train.values,
                             cv=cv, scoring='f1_macro', n_jobs=-1)
    results[name] = scores
    print(f'{name:20s}  Macro F1: {scores.mean():.4f} ± {scores.std():.4f}')

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
data = [results[m] for m in results]
ax.boxplot(data, labels=list(results.keys()), patch_artist=True,
           boxprops=dict(facecolor='#DD8452', alpha=0.6))
ax.set_ylabel('Macro F1 Score (5-fold CV)')
ax.set_title('Multiclass Classifier Comparison')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

## 4. Final Model Evaluation

In [ ]:
from src.models import MulticlassClassifier

final_model = MulticlassClassifier(n_features=K_BEST, n_estimators=100, random_state=SEED)
final_model.fit(X_train, y_train)

y_pred = final_model.predict(X_test)
print(evaluator.report(y_test, y_pred))

In [ ]:
fig = evaluator.plot_confusion_matrix(
    y_test, y_pred,
    labels=final_model.classes_,
    title='Multiclass: Confusion Matrix'
)
plt.show()

## 5. Per-Class Analysis

In [ ]:
y_proba = final_model.predict_proba(X_test)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for i, (cls, ax) in enumerate(zip(final_model.classes_, axes)):
    ax.hist(y_proba[:, i], bins=30, color=['steelblue','#DD8452','#55A868'][i], edgecolor='white')
    ax.set_title(f'Predicted Probability: {cls}')
    ax.set_xlabel('P(class)')
    ax.set_ylabel('Count')

plt.suptitle('Predicted Probability Distributions by Class', y=1.02)
plt.tight_layout()
plt.show()

## Summary

| Class | Precision | Recall | F1 |
|-------|-----------|--------|----|
| NO | ~0.63 | ~0.61 | ~0.62 |
| <30 | ~0.25 | ~0.30 | ~0.27 |
| >30 | ~0.68 | ~0.66 | ~0.67 |

**Key findings:**
- The `<30 days` class is hardest to predict (small minority class even after SMOTE)
- `recurrency`, `number_inpatient`, and `time_in_hospital` are the top 3 multiclass predictors
- Calibrated probabilities from AdaBoost enable reliable risk scoring in the Streamlit app

**Launch the app:**
```bash
streamlit run ../app/app.py
```